# Import Moules

In [1]:
import numpy as np
import pandas as pd
import joblib as jb

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Import Dataset

In [2]:
df = pd.read_csv("../KG_Student_Artifact/KG_Student_Cleaned.csv")
df.head()


,Age,Course,Female,CGPA,Stress_Level,Depression_Score,Anxiety_Score,Sleep_Quality,Physical_Activity,Diet_Quality,Social_Support,Relationship_Status,Substance_Use,Family_History,Chronic_Illness,Financial_Stress,Extracurricular_Activity,Residence_Type
0,25,5,0,3.56,3,3,2,2,1,2,1,2,2,0,0,2,1,0
1,24,2,1,2.44,0,3,0,1,0,1,0,0,1,0,0,3,0,0
2,19,4,1,3.74,4,0,3,2,0,1,1,1,2,0,0,4,2,0
3,18,4,0,3.40,3,3,4,2,0,1,2,2,2,0,1,0,2,0
4,21,0,1,3.35,2,4,3,2,1,2,2,0,2,0,0,5,1,1


# Separate Features and Target

In [3]:
df.columns

Index(['Age', 'Course', 'Female', 'CGPA', 'Stress_Level', 'Depression_Score',
       'Anxiety_Score', 'Sleep_Quality', 'Physical_Activity', 'Diet_Quality',
       'Social_Support', 'Relationship_Status', 'Substance_Use',
       'Family_History', 'Chronic_Illness', 'Financial_Stress',
       'Extracurricular_Activity', 'Residence_Type'],
      dtype='object')

In [4]:
X = df.drop(columns = ['Stress_Level', 'Depression_Score', 'Anxiety_Score'])
Y = df['Anxiety_Score']

# Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.2, stratify=Y, random_state=42,)

# Parameter Grid

GridSearchCV tries every combination of these values
| # | Parameter           | Description                                  |
|---|---------------------|----------------------------------------------|
| 1 | `n_estimators`      | Number of trees                              |
| 2 | `max_depth`         | Max tree depth (`None` = unlimited)          |
| 3 | `min_samples_leaf`  | Min samples allowed in a leaf                |
| 4 | `min_samples_split` | Min samples needed to split a node           |
| 5 | `max_features`      | Features considered at each split            |

In [6]:
param_grid = {
    "n_estimators": [200, 500],           
    "max_depth": [None, 10, 20],          
    "min_samples_leaf": [1, 2, 4],        
    "min_samples_split": [2, 10],         
    "max_features": ["sqrt", 0.5, None],  
}

# Model Object

In [7]:
rf = RandomForestClassifier(random_state=42, class_weight="balanced",)

# Grid Search CV Setup

| # | Parameter           | Description                              |
|---|---------------------|------------------------------------------|
| 1 | `estimator`         | Model to tune                            |
| 2 | `param_grid`        | Options to try                           |
| 3 | `cv`                | Cross-validation                         |
| 4 | `scoring`           | Metric suited to multiclass              |
| 5 | `n_jobs`            | Use all CPU cores                        |
| 6 | `verbose`           | Show progress                            |

In [8]:
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)

# Fit GridSearchCV

In [9]:
grid_rf.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 0.5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 10], ...}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,500


# Best parameters and CV score

In [10]:
print("Best parameters:", grid_rf.best_params_)
print("Best CV f1_macro:", grid_rf.best_score_)

Best parameters: {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 500}
Best CV f1_macro: 0.20444840065562336


# Final evaluation on the test set

In [11]:
y_pred = grid_rf.best_estimator_.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred)) 
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.1937097927090779
              precision    recall  f1-score   support

           0       0.19      0.26      0.22       247
           1       0.16      0.19      0.18       253
           2       0.20      0.14      0.17       259
           3       0.22      0.18      0.20       285
           4       0.19      0.18      0.19       181
           5       0.20      0.22      0.21       174

    accuracy                           0.19      1399
   macro avg       0.20      0.20      0.19      1399
weighted avg       0.20      0.19      0.19      1399

[[64 82 40 30 13 18]
 [63 48 37 50 25 30]
 [64 49 36 47 28 35]
 [67 50 33 51 40 44]
 [35 34 18 31 33 30]
 [40 29 12 20 34 39]]


# See all combinations tried

In [12]:
results = pd.DataFrame(grid_rf.cv_results_)
print(
    results[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
    .head(10)                           
)

                                               params  mean_test_score  \
99  {'max_depth': 20, 'max_features': None, 'min_s...         0.204448   
57  {'max_depth': 10, 'max_features': 0.5, 'min_sa...         0.204204   
98  {'max_depth': 20, 'max_features': None, 'min_s...         0.204033   
27  {'max_depth': None, 'max_features': None, 'min...         0.204028   
69  {'max_depth': 10, 'max_features': None, 'min_s...         0.203630   
88  {'max_depth': 20, 'max_features': 0.5, 'min_sa...         0.203298   
59  {'max_depth': 10, 'max_features': 0.5, 'min_sa...         0.202948   
26  {'max_depth': None, 'max_features': None, 'min...         0.202932   
93  {'max_depth': 20, 'max_features': 0.5, 'min_sa...         0.202612   
48  {'max_depth': 10, 'max_features': 0.5, 'min_sa...         0.202567   

    std_test_score  rank_test_score  
99        0.005271                1  
57        0.008442                2  
98        0.011154                3  
27        0.005541               

# Feature importances

In [13]:
importances = pd.Series(grid_rf.best_estimator_.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

CGPA                        0.241869
Age                         0.146412
Financial_Stress            0.090920
Course                      0.072991
Social_Support              0.052181
Extracurricular_Activity    0.051005
Diet_Quality                0.050706
Relationship_Status         0.050087
Residence_Type              0.049737
Physical_Activity           0.048510
Sleep_Quality               0.045728
Female                      0.031153
Family_History              0.028402
Substance_Use               0.027528
Chronic_Illness             0.012771
dtype: float64


# Save Model

In [14]:
jb.dump(grid_rf.best_estimator_, "../KG_Student_Artifact/KG_Student_Anxiety.joblib")

['../KG_Student_Artifact/KG_Student_Anxiety.joblib']